# 旅さき: Gemma 4 に LoRA をかけて、GGUF にして、採点する（#434）

**使いかた**: 上のメニューの「ランタイム」→「ランタイムのタイプを変更」で **L4 GPU**（Colab Pro）か **T4 GPU**（無料）を選び、「ランタイム」→「**すべてのセルを実行**」を押すだけです。
L4 で 40〜60分、T4 で 1.5〜2時間ほどで、いちばん下のセルに **貼り付ける結果** が出ます。それをそのままチャットに貼ってください。

やること（`training/pipeline.sh` と同じ順です）:

1. Node.js 22・Unsloth・Ollama を入れます（ログインも鍵も要りません。Gemma 4 は Apache-2.0 で、同意の手続きもありません）
2. 学習の前の Gemma 4 E2B を採点します（比べるための基準）
3. 学習データを書き出し、LoRA（r=64）をかけます
4. GGUF（Q4_K_M、約 3GB）にして、Ollama に `tabisaki-gemma` として入れます
5. 採点して、合否を出します（`training/eval_gate.mjs`）
6. （任意）Hugging Face Hub に置きます。既定では置きません（下の `PUSH_TO_HF`）

お金: 無料の T4 でも回ります。Colab Pro（月 $9.99）なら L4 で速く回せます。途中で止まったら、もう一度「すべてのセルを実行」を押してください。

In [ ]:
# ── 設定（ふつうは変えなくて大丈夫です）──
BASE = "unsloth/gemma-4-E2B-it"   # 元のモデル。大きくするなら "unsloth/gemma-4-E4B-it"
OLLAMA_BASE = "gemma4:e2b"        # 学習の前の採点に使う Ollama のモデル（BASE と同じ大きさに）
TAG = "colab-e2b-r64"             # 結果のフォルダの名前（runs/<TAG>）
EPOCHS = 1                        # 何周学ぶか
EVAL_LIMIT = 0                    # 採点する問題の数（0 = 全部）
EXPORT_ARGS = ""                  # 学習データの範囲。"--noncommercial --verify=high" で約3.8万スポット（重みは非商用・非公開に）
PUSH_TO_HF = False                # True にすると Hugging Face Hub に置きます（合格のときだけ）
HF_REPO = ""                      # 置き場所（<ユーザー名>/tabisaki-gemma-4-e2b）。左の鍵のマークに HF_TOKEN を
REPO = "https://github.com/shitianliang1000-alt/tabisaki-app.git"
BRANCHES = ["main", "claude/project-thread-ep4eea"]  # 先にあるほうを使います（まとめの PR が入る前でも動くように）

In [ ]:
# GPU の確認
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("⚠️ GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で L4 か T4 を選んでから、もう一度すべて実行してください。")

In [ ]:
%%bash
# Node.js 22 を入れます
set -e
if ! node --version 2>/dev/null | grep -q '^v2[2-9]'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null
  apt-get install -y nodejs > /dev/null
fi
node --version

In [ ]:
# リポジトリを落とします（training/train_lora.py があるブランチを使います）
import os, subprocess
os.chdir("/content")
subprocess.run(["rm", "-rf", "tabisaki"])
used = None
for b in BRANCHES:
    r = subprocess.run(["git", "clone", "-q", "--depth=1", "-b", b, REPO, "tabisaki"], capture_output=True, text=True)
    if r.returncode == 0 and os.path.exists("tabisaki/training/train_lora.py"):
        used = b
        break
    subprocess.run(["rm", "-rf", "tabisaki"])
if not used:
    raise SystemExit("リポジトリを落とせませんでした。ネットにつながっているか確かめてください。")
os.chdir("/content/tabisaki")
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("ブランチ:", used, " コミット:", COMMIT)

In [ ]:
%%bash
# Unsloth（学習と GGUF づくり）を入れます。数分かかります
pip install -q unsloth huggingface_hub 2>&1 | tail -2
python -c "import unsloth, torch; print('unsloth', unsloth.__version__, 'torch', torch.__version__)"

In [ ]:
%%bash
# Ollama を入れて、裏で動かします
set -e
if ! command -v ollama > /dev/null; then
  apt-get install -y zstd > /dev/null 2>&1 || true
  curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
fi
pkill -f "ollama serve" 2>/dev/null || true
OLLAMA_CONTEXT_LENGTH=8192 OLLAMA_KEEP_ALIVE=5m nohup ollama serve > /content/ollama.log 2>&1 &
for i in $(seq 1 60); do curl -s localhost:11434/api/version > /dev/null && break; sleep 1; done
curl -s localhost:11434/api/version; echo

In [ ]:
# 1. 学習の前の Gemma を採点します（比べるための基準）
import subprocess, os
os.makedirs("runs", exist_ok=True)
print(subprocess.run(["node", "tools/export_training_data.mjs", *EXPORT_ARGS.split()], capture_output=True, text=True).stdout[-600:])
print(subprocess.run(["ollama", "pull", OLLAMA_BASE], capture_output=True, text=True).stdout[-200:])
r = subprocess.run(["node", "tools/model_eval.mjs", "--url=http://localhost:11434", f"--model={OLLAMA_BASE}",
                    f"--limit={EVAL_LIMIT}", "--out=runs/base-eval.json"], capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-2000:])
subprocess.run(["ollama", "stop", OLLAMA_BASE], capture_output=True)  # GPU のメモリを学習に空けます

In [ ]:
# 2〜5. 学習 → GGUF → Ollama → 採点 → 合否（training/pipeline.sh）
import subprocess, os
env = dict(os.environ, TAG=TAG, EXPORT_ARGS=EXPORT_ARGS, BASE=BASE, EPOCHS=str(EPOCHS), EVAL_LIMIT=str(EVAL_LIMIT),
           BASE_EVAL="runs/base-eval.json" if os.path.exists("runs/base-eval.json") else "")
p = subprocess.Popen(["bash", "training/pipeline.sh"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tail = []
for line in p.stdout:
    if "it/s]" in line or "s/it]" in line:
        continue  # 進み具合の棒は省きます
    print(line, end="")
    tail = (tail + [line])[-80:]
PIPE_CODE = p.wait()
print("\n終了コード:", PIPE_CODE, "（0 = 合格、1 = 不合格、それ以外 = 途中で止まった）")

In [ ]:
# 試しに1問（学習したモデルが Gemma 4 の会話の形で答えるか）
import subprocess
for q in ["大通公園について教えてください。", "「ほげほげ岬」ってどんなところ？"]:
    r = subprocess.run(["ollama", "run", "tabisaki-gemma", q], capture_output=True, text=True, timeout=300)
    print("Q:", q, "\nA:", (r.stdout or r.stderr).strip()[:400], "\n")

In [ ]:
# 6.（任意）Hugging Face Hub に置きます。PUSH_TO_HF = True で、合格のときだけ
import subprocess, os
RUN = f"runs/{TAG}"
args = ["python", "training/publish_hf.py", RUN, "--repo", HF_REPO or "you/tabisaki-gemma-4-e2b"]
if PUSH_TO_HF and HF_REPO:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN が読めませんでした:", e)
    args.append("--push")
if EXPORT_ARGS:
    args.append("--noncommercial")  # 非公開に限って上げます
r = subprocess.run(args, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-1000:])

In [ ]:
# 貼り付ける結果（このセルの出力を、まるごとチャットに貼ってください）
import json, os, subprocess, datetime, glob
RUN = f"runs/{TAG}"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("==== ここから貼り付け ====")
print(f"## Gemma 4 LoRA の一巡（{datetime.date.today().isoformat()}、{gpu}、コミット {COMMIT}）")
if os.path.exists(f"{RUN}/run.json"):
    run = json.load(open(f"{RUN}/run.json"))
    print("\n```json")
    print(json.dumps({k: run.get(k) for k in ("base", "backend", "seconds", "lora", "train", "result")}, ensure_ascii=False, indent=1))
    print("```")
for g in glob.glob(f"{RUN}/gguf/*.gguf"):
    print(f"\nGGUF: {os.path.basename(g)}（{os.path.getsize(g) / 1e9:.2f}GB）")
for name in ("base-eval.json",):
    if os.path.exists(f"runs/{name}"):
        print(f"\n### 学習の前（{OLLAMA_BASE}）\n```json\n" + json.dumps(json.load(open(f'runs/{name}'))['summary'], indent=1) + "\n```")
if os.path.exists(f"{RUN}/gate.md"):
    print("\n### 合否\n" + open(f"{RUN}/gate.md").read())
else:
    print("\n（合否が出ていません。上のセルのエラーを貼ってください）\n" + "".join(tail[-30:]))
print("\n==== ここまで ====")